##IMPORTAÇÃO DE BIBLIOTECAS

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

##ANÁLISE EXPLORATÓRIA

In [ ]:
#leitura do train.csv
df = pd.read_csv("train.csv")
df.describe()

In [ ]:
display(df.head())

In [ ]:
df.info()

## PRÉ-PROCESSAMENTO DOS DADOS

In [ ]:
def preprocess_covid_data(df, is_train=True, encoders=None):
    df = df.copy()

    # Colunas categóricas principais
    categorical_cols = ['CS_SEXO', 'CS_RACA', 'CS_ZONA', 'SG_UF', 'ID_MN_RESI']

    # Colunas de comorbidades e sintomas (assumindo valores 1=Sim, 2=Não, 9=Ignorado)
    binary_cols = ['DIABETES', 'OBESIDADE', 'RENAL', 'HEPATICA', 'IMUNODEPRE',
                   'FEBRE', 'TOSSE', 'GARGANTA', 'DISPNEIA', 'DESC_RESP',
                   'DIARREIA', 'VOMITO', 'FADIGA', 'VACINA_COV', 'FADIGA']

    # Processa colunas binárias (1->1, 2->0, outros->0)
    for col in binary_cols:
        if col in df.columns:
            df[col] = (df[col] == 1).astype(int)

    # Processa idade
    if 'NU_IDADE_N' in df.columns:
        df['NU_IDADE_N'] = df['NU_IDADE_N'].fillna(df['NU_IDADE_N'].median())
        df['idoso'] = (df['NU_IDADE_N'] >= 60).astype(int)

    # Processa saturação
    if 'SATURACAO' in df.columns:
        df['SATURACAO'] = pd.to_numeric(df['SATURACAO'], errors='coerce')
        df['saturacao_baixa'] = (df['SATURACAO'] < 95).astype(int)
        df['SATURACAO'] = df['SATURACAO'].fillna(98)

    # Educação
    if 'CS_ESCOL_N' in df.columns:
        df['CS_ESCOL_N'] = df['CS_ESCOL_N'].fillna(0)
        df['baixa_educacao'] = (df['CS_ESCOL_N'] <= 3).astype(int)


    # Features criadas soma de comorbidades e sintomas
    comorbidades = [c for c in ['DIABETES', 'OBESIDADE', 'RENAL', 'HEPATICA', 'IMUNODEPRE'] if c in df.columns]
    sintomas = [c for c in ['FEBRE', 'TOSSE', 'GARGANTA', 'DISPNEIA', 'DESC_RESP', 'DIARREIA', 'VOMITO', 'FADIGA'] if c in df.columns]

    df['total_comorbidades'] = df[comorbidades].sum(axis=1)
    df['total_sintomas'] = df[sintomas].sum(axis=1)

    if 'NU_IDADE_N' in df.columns:
        df['risco_idade_comorb'] = df['NU_IDADE_N'] * df['total_comorbidades']

    # Encoding de categoricas
    if is_train:
        encoders = {}
        for col in categorical_cols:
            if col in df.columns:
                le = LabelEncoder()
                df[col] = df[col].fillna('UNKNOWN')
                df[col] = le.fit_transform(df[col].astype(str))
                encoders[col] = le
    else:
        for col in categorical_cols:
            if col in df.columns and col in encoders:
                df[col] = df[col].fillna('UNKNOWN')
                # Trata valores não vistos no treino
                df[col] = df[col].astype(str)
                known_values = set(encoders[col].classes_)
                df[col] = df[col].apply(lambda x: x if x in known_values else 'UNKNOWN')
                df[col] = encoders[col].transform(df[col])

    # Remove colunas desnecessárias
    cols_to_drop = ['DT_NOTIFIC', 'DT_SIN_PRI', 'SEM_PRI']
    for col in cols_to_drop:
       if col in df.columns:
          df.drop(col, axis=1, inplace=True)

    # Preenche valores faltantes
    for col in df.columns:
        if col != 'EVOLUCAO':  # Não preenche o target
            if df[col].dtype in ['object']:
                df[col] = df[col].fillna('UNKNOWN')
            else:
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

    print(f"Shape: {df.shape}, Nulos: {df.isnull().sum().sum()}")
    return df, encoders


##BALANCEAMENTO DOS DADOS



In [ ]:
def balance_data(X, y, method='undersample', random_state=42):
    # Combina X e y para facilitar o resampling
    df_combined = pd.concat([X.reset_index(drop=True), y.reset_index(drop=True)], axis=1)

    # Separa por classe
    majority_class = df_combined[df_combined['EVOLUCAO'] == 0]
    minority_class = df_combined[df_combined['EVOLUCAO'] == 1]

    print(f"Antes do balanceamento:")
    print(f"  Classe 0 (alta): {len(majority_class)}")
    print(f"  Classe 1 (óbito): {len(minority_class)}")
    print(f"  Ratio: {len(minority_class)/len(majority_class):.3f}")

    if method == 'undersample':
        # Reduz a classe majoritária
        majority_downsampled = resample(
            majority_class,
            replace=False,
            n_samples=len(minority_class),
            random_state=random_state
        )
        df_balanced = pd.concat([majority_downsampled, minority_class])

    elif method == 'oversample':
        # Aumenta a classe minoritária
        minority_upsampled = resample(
            minority_class,
            replace=True,
            n_samples=len(majority_class),
            random_state=random_state
        )
        df_balanced = pd.concat([majority_class, minority_upsampled])

    elif method == 'mixed':
        # Abordagem mista: reduz majoritária e aumenta minoritária
        target_size = int((len(majority_class) + len(minority_class)) * 0.2)  # 30% do total

        majority_downsampled = resample(
            majority_class,
            replace=False,
            n_samples=target_size,
            random_state=random_state
        )
        minority_upsampled = resample(
            minority_class,
            replace=True,
            n_samples=target_size,
            random_state=random_state
        )
        df_balanced = pd.concat([majority_downsampled, minority_upsampled])

    # Embaralha os dados
    df_balanced = df_balanced.sample(frac=1, random_state=random_state).reset_index(drop=True)

    # Separa X e y novamente
    X_balanced = df_balanced.drop(['EVOLUCAO'], axis=1)
    y_balanced = df_balanced['EVOLUCAO']

    print(f"Após balanceamento ({method}):")
    print(f"  Classe 0 (alta): {sum(y_balanced == 0)}")
    print(f"  Classe 1 (óbito): {sum(y_balanced == 1)}")
    print(f"  Total de amostras: {len(y_balanced)}")
    print(f"  Novo ratio: {sum(y_balanced == 1)/sum(y_balanced == 0):.3f}")

    return X_balanced, y_balanced

## TREINO E VALIDAÇÃO DO MODELO

In [ ]:
def train_covid_model(df_train, balance_method='mixed', n_folds=3, random_state=42):


    print("Preprocessando dados de treino...")
    df_processed, encoders = preprocess_covid_data(df_train, is_train=True)

    # Separa features e target
    X = df_processed.drop(['EVOLUCAO'], axis=1)
    y = df_processed['EVOLUCAO']

    print(f"Features: {X.shape[1]}, Amostras: {X.shape[0]}")
    print(f"Distribuição target original: {y.value_counts().to_dict()}")

    # Aplica balanceamento
    X_balanced, y_balanced = balance_data(X, y, method=balance_method, random_state=random_state)

    # Modelo com class_weight para extra robustez
    model = HistGradientBoostingClassifier(
        max_iter=150,
        learning_rate=0.1,
        max_depth=10,
        min_samples_leaf=20,
        l2_regularization=0.1,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20,
        random_state=random_state
    )

    # K-Fold nos dados balanceados
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)

    aucs = []
    f1s = []

    print(f"\\nIniciando {n_folds}-Fold Cross Validation nos dados balanceados...")

    for fold, (train_idx, val_idx) in enumerate(skf.split(X_balanced, y_balanced), 1):
        print(f"Fold {fold}/{n_folds}")

        X_train_fold, X_val_fold = X_balanced.iloc[train_idx], X_balanced.iloc[val_idx]
        y_train_fold, y_val_fold = y_balanced.iloc[train_idx], y_balanced.iloc[val_idx]

        # Padronização
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_fold)
        X_val_scaled = scaler.transform(X_val_fold)

        # Treino
        model.fit(X_train_scaled, y_train_fold)

        # Predição
        y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
        y_pred_binary = (y_pred_proba > 0.5).astype(int)

        # Métricas
        auc = roc_auc_score(y_val_fold, y_pred_proba)
        f1 = f1_score(y_val_fold, y_pred_binary)

        aucs.append(auc)
        f1s.append(f1)

        print(f"  AUC: {auc:.4f}, F1: {f1:.4f}")

    # Treina modelo final nos dados balanceados
    print("Treinando modelo final...")
    scaler_final = StandardScaler()
    X_scaled_final = scaler_final.fit_transform(X_balanced)
    model.fit(X_scaled_final, y_balanced)

    print(f"\\nCV Results - AUC: {np.mean(aucs):.4f}±{np.std(aucs):.4f}")
    print(f"CV Results - F1:  {np.mean(f1s):.4f}±{np.std(f1s):.4f}")

    return model, scaler_final, encoders, X.columns

##PREDIÇÃO NO CONJUNTO DE TESTES

In [ ]:
def predict_covid_test(model, scaler, encoders, feature_cols, df_test):

    print("Preprocessando dados de teste...")
    df_processed, _ = preprocess_covid_data(df_test, is_train=False, encoders=encoders)

    # Alinha colunas com treino
    for col in feature_cols:
        if col not in df_processed.columns:
            df_processed[col] = 0

    X_test = df_processed[feature_cols]
    X_test_scaled = scaler.transform(X_test)

    print("Fazendo predições...")
    probabilities = model.predict_proba(X_test_scaled)[:, 1]

    # Converte para predições binárias (0 ou 1)
    predictions = (probabilities > 0.5).astype(int)

    print(f"Probabilidades - Min: {probabilities.min():.4f}, Max: {probabilities.max():.4f}")
    print(f"Probabilidades - Média: {probabilities.mean():.4f}")
    print(f"Predições binárias:")
    print(f"  - Altas (0): {sum(predictions == 0)} ({sum(predictions == 0)/len(predictions):.1%})")
    print(f"  - Óbitos (1): {sum(predictions == 1)} ({sum(predictions == 1)/len(predictions):.1%})")

    return predictions

##EXECUÇÃO DO PIPELINE

In [4]:
def run_competition_pipeline(df_train, df_test, balance_method='mixed'):
    """Funcao de treino"""

    # Treina modelo com balanceamento
    model, scaler, encoders, feature_cols = train_covid_model(
        df_train,
        balance_method=balance_method,  # 'undersample', 'oversample', ou 'mixed'
        n_folds=3
    )

    # Predições no teste
    test_predictions = predict_covid_test(model, scaler, encoders, feature_cols, df_test)

    # Prepara submissão
    submission = pd.DataFrame({
        'ID': df_test.index,
        'EVOLUCAO': test_predictions
    })

    return submission

# Carregar dados
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

# Opção 1: Método misto
#submission1 = run_competition_pipeline(df_train, df_test, balance_method='mixed')

# # Opção 2: Undersample (mais rápido)
# submission2 = run_competition_pipeline(df_train, df_test, balance_method='undersample')

# Opção 3: Oversample (mais dados)
submission3 = run_competition_pipeline(df_train, df_test, balance_method='oversample')

# Salvar
submission3.to_csv('submission_oversample.csv', index=False)
print("Submissão com predições binárias salva!")

# Análise da submissão
print(f"Total de predições: {len(submission3)}")
print(f"Altas previstas (0): {sum(submission3['EVOLUCAO'] == 0)}")
print(f"Óbitos previstos (1): {sum(submission3['EVOLUCAO'] == 1)}")
print(f"Taxa de óbito prevista: {submission3['EVOLUCAO'].mean():.2%}")

Preprocessando dados de treino...
Shape: (498320, 37), Nulos: 0
Features: 36, Amostras: 498320
Distribuição target original: {0.0: 326286, 1.0: 172034}
Antes do balanceamento:
  Classe 0 (alta): 326286
  Classe 1 (óbito): 172034
  Ratio: 0.527
Após balanceamento (oversample):
  Classe 0 (alta): 326286
  Classe 1 (óbito): 326286
  Total de amostras: 652572
  Novo ratio: 1.000
\nIniciando 3-Fold Cross Validation nos dados balanceados...
Fold 1/3
  AUC: 0.8072, F1: 0.7385
Fold 2/3
  AUC: 0.8067, F1: 0.7386
Fold 3/3
  AUC: 0.8079, F1: 0.7384
Treinando modelo final...
\nCV Results - AUC: 0.8073±0.0005
CV Results - F1:  0.7385±0.0001
Preprocessando dados de teste...
Shape: (124581, 36), Nulos: 0
Fazendo predições...
Probabilidades - Min: 0.0070, Max: 0.9924
Probabilidades - Média: 0.4572
Predições binárias:
  - Altas (0): 67124 (53.9%)
  - Óbitos (1): 57457 (46.1%)
Submissão com predições binárias salva!
Total de predições: 124581
Altas previstas (0): 67124
Óbitos previstos (1): 57457
Taxa d